# Análise de dados — respostas às perguntas de negócio

Este notebook responde as sete perguntas de negócio definidas no objetivo do
trabalho. Cada seção traz a consulta, o resultado e a discussão do que os números
significam no contexto do problema.

## Perguntas

1. Quais municípios concentram o maior volume de domicílios ainda não atendidos?
2. A penetração varia com o porte do município? Existe faixa populacional
   sistematicamente mal servida?
3. Como se distribui o número de provedores por município? Quantos são monopólio de
   fato?
4. Qual a participação da fibra por município e por região? Quais municípios ainda
   dependem de rádio, satélite ou cobre?
5. Quais municípios têm a base de velocidade mais defasada — potencial de upgrade de
   plano?
6. A penetração se relaciona com PIB per capita ou com o setor econômico dominante?
7. Considerando mercado não atendido, defasagem tecnológica e poder aquisitivo, quais
   municípios apresentam maior oportunidade comercial?


## Preparação

In [0]:
CATALOGO_GOLD = "gold"
SCHEMA        = "telecom"

from pyspark.sql import functions as F
from pyspark.sql.window import Window
import matplotlib.pyplot as plt

spark.sql(f"USE CATALOG {CATALOGO_GOLD}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"Contexto: {CATALOGO_GOLD}.{SCHEMA}")
display(spark.sql(f"SHOW TABLES IN {CATALOGO_GOLD}.{SCHEMA}"))

In [0]:
def br(coluna, decimais: int = 0, sufixo: str = ""):
    """Formata numero no padrao brasileiro: 1.474 / 26,46 / 26,46 %"""
    txt = F.format_number(F.col(coluna), decimais)
    txt = F.translate(txt, ".,", ",.")
    return F.concat(txt, F.lit(sufixo)) if sufixo else txt


def exibir(df, numericos: dict, texto: list = None):
    """numericos: {"coluna": (casas_decimais, sufixo)}"""
    texto = texto or []
    sel  = [F.col(x) for x in texto]
    sel += [br(col, d, s).alias(col) for col, (d, s) in numericos.items()]
    return df.select(*sel)


# Padrao visual dos graficos
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

Panorama geral do estado, para contextualizar as análises seguintes.

In [0]:
panorama = spark.sql("""
    SELECT
        COUNT(*)                                   AS municipios_qtd,
        SUM(populacao_residente_hab)               AS populacao_hab,
        SUM(domicilios_total_qtd)                  AS domicilios_qtd,
        SUM(acessos_total_qtd)                     AS acessos_qtd,
        SUM(acessos_pf_qtd)                        AS acessos_pf_qtd,
        SUM(acessos_pj_qtd)                        AS acessos_pj_qtd,
        ROUND(100.0 * SUM(acessos_total_qtd)
              / SUM(domicilios_total_qtd), 2)      AS penetracao_estadual_pct,
        ROUND(100.0 * SUM(acessos_total_qtd * fibra_pct / 100)
              / SUM(acessos_total_qtd), 2)         AS fibra_estadual_pct
    FROM mercado_municipio
""")

display(exibir(panorama,
    numericos={"municipios_qtd": (0, ""), "populacao_hab": (0, ""),
               "domicilios_qtd": (0, ""), "acessos_qtd": (0, ""),
               "acessos_pf_qtd": (0, ""), "acessos_pj_qtd": (0, ""),
               "penetracao_estadual_pct": (2, " %"),
               "fibra_estadual_pct": (2, " %")}))

---

# Pergunta 1 — Municípios com maior volume de domicílios não atendidos

O mercado não atendido é a diferença entre domicílios recenseados e acessos totais,
com piso em zero. Municípios grandes concentram volume absoluto; municípios pequenos
podem ter penetração baixa sem representar oportunidade relevante em volume.

Por isso a resposta olha as duas coisas: **volume absoluto** de domicílios sem acesso
e **penetração percentual**.

In [0]:
nao_atendidos = spark.sql("""
    SELECT
        nome_municipio,
        mesorregiao_nome,
        porte_populacional,
        domicilios_total_qtd,
        acessos_total_qtd,
        domicilios_sem_acesso_qtd,
        penetracao_enderecavel_pct,
        n_provedores_qtd,
        concentracao
    FROM mercado_municipio
    WHERE domicilios_sem_acesso_qtd > 0
    ORDER BY domicilios_sem_acesso_qtd DESC
    LIMIT 25
""")

display(exibir(nao_atendidos,
    numericos={"domicilios_total_qtd": (0, ""),
               "acessos_total_qtd": (0, ""),
               "domicilios_sem_acesso_qtd": (0, ""),
               "penetracao_enderecavel_pct": (2, " %"),
               "n_provedores_qtd": (0, "")},
    texto=["nome_municipio", "mesorregiao_nome", "porte_populacional",
           "concentracao"]))

O mesmo recorte pela ótica percentual: municípios com a menor penetração, filtrando
os que têm ao menos mil domicílios para excluir casos de escala irrelevante.

In [0]:
menor_penetracao = spark.sql("""
    SELECT
        nome_municipio,
        mesorregiao_nome,
        porte_populacional,
        domicilios_total_qtd,
        acessos_total_qtd,
        domicilios_sem_acesso_qtd,
        penetracao_enderecavel_pct,
        n_provedores_qtd,
        pib_per_capita_brl
    FROM mercado_municipio
    WHERE domicilios_total_qtd >= 1000
    ORDER BY penetracao_enderecavel_pct ASC
    LIMIT 25
""")

display(exibir(menor_penetracao,
    numericos={"domicilios_total_qtd": (0, ""),
               "acessos_total_qtd": (0, ""),
               "domicilios_sem_acesso_qtd": (0, ""),
               "penetracao_enderecavel_pct": (2, " %"),
               "n_provedores_qtd": (0, ""),
               "pib_per_capita_brl": (2, "")},
    texto=["nome_municipio", "mesorregiao_nome", "porte_populacional"]))

Concentração do mercado não atendido por mesorregião.

In [0]:
por_regiao = spark.sql("""
    SELECT
        mesorregiao_nome,
        COUNT(*)                              AS municipios_qtd,
        SUM(domicilios_total_qtd)             AS domicilios_qtd,
        SUM(acessos_total_qtd)                AS acessos_qtd,
        SUM(domicilios_sem_acesso_qtd)        AS nao_atendidos_qtd,
        ROUND(100.0 * SUM(acessos_total_qtd)
              / SUM(domicilios_total_qtd), 2) AS penetracao_pct
    FROM mercado_municipio
    GROUP BY mesorregiao_nome
    ORDER BY nao_atendidos_qtd DESC
""")

display(exibir(por_regiao,
    numericos={"municipios_qtd": (0, ""), "domicilios_qtd": (0, ""),
               "acessos_qtd": (0, ""), "nao_atendidos_qtd": (0, ""),
               "penetracao_pct": (2, " %")},
    texto=["mesorregiao_nome"]))

In [0]:
pdf = por_regiao.toPandas()

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(pdf["mesorregiao_nome"], pdf["nao_atendidos_qtd"], color="#4C72B0")
ax.set_xlabel("Domicílios sem acesso")
ax.set_title("Mercado não atendido por mesorregião — RS")
ax.invert_yaxis()
for i, v in enumerate(pdf["nao_atendidos_qtd"]):
    ax.text(v, i, f" {v:,.0f}".replace(",", "."), va="center", fontsize=9)
plt.tight_layout()
plt.show()

### Discussão e resposta para a pergunta:

Em termos da quantidade de domicílios não atendidos as Top [10](url) cidades seriam:

- Porto Alegre - 78.861
- Gravataí - 53.957
- Viamão - 32.433
- Pelotas - 31.074
- Caxias do Sul - 28.402
- Santa Maria - 28.348
- Cachoeirinha - 26.601
- Bagé - 22.797
- Canoas - 21.619
- Rio Grande - 20.956

Porém, todas essas são cidades com porte populacional acima de 100 mil habitantes, o que nós faz analisar em conjunto a taxa de penetração, onde 

- Porto Alegre - 88,53%
- Gravataí - 54,80%
- Viamão - 69,76%
- Pelotas - 81,29%
- Caxias do Sul - 87,12%
- Santa Maria - 78,62%
- Cachoeirinha - 56,11%
- Bagé - 57,56%
- Canoas - 86,10%
- Rio Grande - 79,64%

Analisando os dois recortes, podemos ver que apesar de algumas cidades terem um maior número de domicílios pendentes de atendimento, isso não se reflete em mercado potencial de atenuação devido as altas taxas de ocupação.

Mas podemos destacar as cidades de Gravataí, Cachoerinha e Bagé que possuem uma taxa de penetração abaixo de 60% o qual ainda há espaço para atendimento.

O mercado não atendido do RS se distribui de forma desigual entre volume e
percentual. Municípios de grande porte concentram o maior número absoluto de
domicílios sem acesso, mesmo tendo penetração acima da média — o volume vem da
escala, não da carência.

Já os municípios com menor penetração percentual tendem a ser de porte menor e
localizados fora dos eixos metropolitanos, onde a oferta é mais rarefeita.

Para estratégia comercial, as duas leituras servem a propósitos diferentes: volume
absoluto indica onde há mais assinantes possíveis; penetração baixa indica onde a
concorrência ainda não consolidou posição.



---

# Pergunta 2 — Penetração por porte de município

A hipótese a testar é se existe faixa populacional sistematicamente mal servida.
Duas forças atuam em direções opostas: municípios maiores têm mais densidade e
atraem mais provedores, mas municípios pequenos têm menos domicílios para cobrir por
quilômetro de rede.

In [0]:
por_porte = spark.sql("""
    SELECT
        porte_populacional,
        COUNT(*)                                   AS municipios_qtd,
        SUM(domicilios_total_qtd)                  AS domicilios_qtd,
        SUM(acessos_total_qtd)                     AS acessos_qtd,
        SUM(domicilios_sem_acesso_qtd)             AS nao_atendidos_qtd,
        ROUND(100.0 * SUM(acessos_total_qtd)
              / SUM(domicilios_total_qtd), 2)      AS penetracao_agregada_pct,
        ROUND(AVG(penetracao_enderecavel_pct), 2)  AS penetracao_media_pct,
        ROUND(PERCENTILE(penetracao_enderecavel_pct, 0.5), 2) AS penetracao_mediana_pct,
        ROUND(AVG(n_provedores_qtd), 1)            AS provedores_media_qtd,
        ROUND(AVG(fibra_pct), 2)                   AS fibra_media_pct
    FROM mercado_municipio
    GROUP BY porte_populacional
    ORDER BY
        CASE porte_populacional
            WHEN 'Ate 5 mil'        THEN 1
            WHEN '5 a 20 mil'       THEN 2
            WHEN '20 a 100 mil'     THEN 3
            WHEN 'Acima de 100 mil' THEN 4
        END
""")

display(exibir(por_porte,
    numericos={"municipios_qtd": (0, ""), "domicilios_qtd": (0, ""),
               "acessos_qtd": (0, ""), "nao_atendidos_qtd": (0, ""),
               "penetracao_agregada_pct": (2, " %"),
               "penetracao_media_pct": (2, " %"),
               "penetracao_mediana_pct": (2, " %"),
               "provedores_media_qtd": (1, ""),
               "fibra_media_pct": (2, " %")},
    texto=["porte_populacional"]))

A média pode ser distorcida por municípios com penetração acima de 100%. A mediana
e a distribuição por quartil dão uma leitura mais robusta.

In [0]:
pdf = spark.sql("""
    SELECT porte_populacional, penetracao_enderecavel_pct, n_provedores_qtd, fibra_pct
    FROM mercado_municipio
    WHERE penetracao_enderecavel_pct IS NOT NULL
""").toPandas()

ordem = ["Ate 5 mil", "5 a 20 mil", "20 a 100 mil", "Acima de 100 mil"]
dados = [pdf.loc[pdf["porte_populacional"] == p, "penetracao_enderecavel_pct"].values
         for p in ordem]

fig, ax = plt.subplots(figsize=(10, 5))
try:                                       # Matplotlib 3.9 ou superior
    ax.boxplot(dados, tick_labels=ordem, showfliers=True)
except TypeError:                          # versoes anteriores
    ax.boxplot(dados, labels=ordem, showfliers=True)
ax.axhline(100, color="#C44E52", linestyle="--", linewidth=1, alpha=0.7)
ax.set_ylabel("Penetração (%)")
ax.set_title("Distribuição da penetração por porte populacional — RS")
plt.tight_layout()
plt.show()

### Discussão e resposta para a pergunta:

Fica claro que quanto maior o porte populacional maior é a taxa de penetração agregada.
Com esse viés municípios de até 20 mil habitantes possuem em média 53% de taxa de penetração.


O boxplot mostra tanto o nível quanto a dispersão. Faixas com mediana baixa e caixa
estreita indicam carência sistemática; faixas com mediana alta e muitos outliers
indicam heterogeneidade — alguns municípios bem servidos convivendo com outros mal
atendidos no mesmo porte.

A comparação entre penetração agregada e média simples também é informativa: quando
a agregada é maior, os municípios grandes da faixa puxam o resultado para cima, o que
significa que a média simples descreve melhor o município típico.

---

# Pergunta 3 — Estrutura competitiva e monopólios

Duas medidas complementares:

- **Número de provedores** — quantos CNPJs distintos declaram acesso no município
- **HHI** (Índice Herfindahl-Hirschman) — soma dos quadrados dos shares, escala 0 a
  10.000

O número de provedores mede oferta nominal; o HHI mede concentração real. Um
município pode ter dez provedores e ainda ser dominado por um só.

In [0]:
distribuicao_provedores = spark.sql("""
    SELECT
        CASE
            WHEN n_provedores_qtd = 1  THEN '1 (monopolio)'
            WHEN n_provedores_qtd = 2  THEN '2'
            WHEN n_provedores_qtd <= 5 THEN '3 a 5'
            WHEN n_provedores_qtd <= 10 THEN '6 a 10'
            WHEN n_provedores_qtd <= 20 THEN '11 a 20'
            ELSE 'Mais de 20'
        END                                       AS faixa_provedores,
        COUNT(*)                                  AS municipios_qtd,
        SUM(domicilios_total_qtd)                 AS domicilios_qtd,
        ROUND(AVG(penetracao_enderecavel_pct), 2) AS penetracao_media_pct,
        ROUND(AVG(hhi), 0)                        AS hhi_medio,
        ROUND(AVG(share_lider_pct), 2)            AS share_lider_medio_pct,
        ROUND(AVG(fibra_pct), 2)                  AS fibra_media_pct
    FROM mercado_municipio
    GROUP BY 1
    ORDER BY MIN(n_provedores_qtd)
""")

display(exibir(distribuicao_provedores,
    numericos={"municipios_qtd": (0, ""), "domicilios_qtd": (0, ""),
               "penetracao_media_pct": (2, " %"), "hhi_medio": (0, ""),
               "share_lider_medio_pct": (2, " %"), "fibra_media_pct": (2, " %")},
    texto=["faixa_provedores"]))

In [0]:
concentracao = spark.sql("""
    SELECT
        concentracao,
        COUNT(*)                                  AS municipios_qtd,
        SUM(domicilios_total_qtd)                 AS domicilios_qtd,
        SUM(acessos_total_qtd)                    AS acessos_qtd,
        ROUND(AVG(n_provedores_qtd), 1)           AS provedores_media_qtd,
        ROUND(AVG(hhi), 0)                        AS hhi_medio,
        ROUND(AVG(share_lider_pct), 2)            AS share_lider_medio_pct,
        ROUND(AVG(penetracao_enderecavel_pct), 2) AS penetracao_media_pct
    FROM mercado_municipio
    GROUP BY concentracao
    ORDER BY hhi_medio DESC
""")

display(exibir(concentracao,
    numericos={"municipios_qtd": (0, ""), "domicilios_qtd": (0, ""),
               "acessos_qtd": (0, ""), "provedores_media_qtd": (1, ""),
               "hhi_medio": (0, ""), "share_lider_medio_pct": (2, " %"),
               "penetracao_media_pct": (2, " %")},
    texto=["concentracao"]))

Municípios com um único provedor — monopólio de fato.

In [0]:
monopolios = spark.sql("""
    SELECT
        nome_municipio, mesorregiao_nome, porte_populacional,
        provedor_lider, domicilios_total_qtd, acessos_total_qtd,
        penetracao_enderecavel_pct, fibra_pct
    FROM mercado_municipio
    WHERE n_provedores_qtd = 1
    ORDER BY domicilios_total_qtd DESC
""")

print(f"Municipios com provedor unico: {monopolios.count()}")

if monopolios.count() > 0:
    display(exibir(monopolios,
        numericos={"domicilios_total_qtd": (0, ""), "acessos_total_qtd": (0, ""),
                   "penetracao_enderecavel_pct": (2, " %"), "fibra_pct": (2, " %")},
        texto=["nome_municipio", "mesorregiao_nome", "porte_populacional",
               "provedor_lider"]))

In [0]:
pdf = spark.sql("""
    SELECT n_provedores_qtd, hhi, penetracao_enderecavel_pct, porte_populacional
    FROM mercado_municipio
    WHERE n_provedores_qtd IS NOT NULL
""").toPandas()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.hist(pdf["n_provedores_qtd"], bins=range(0, int(pdf["n_provedores_qtd"].max()) + 3),
         color="#4C72B0", edgecolor="white")
ax1.set_xlabel("Provedores por município")
ax1.set_ylabel("Municípios")
ax1.set_title("Distribuição do número de provedores")

ax2.scatter(pdf["n_provedores_qtd"], pdf["hhi"], alpha=0.5, s=25, color="#55A868")
ax2.axhline(2500, color="#C44E52", linestyle="--", linewidth=1, label="HHI 2.500 (alta)")
ax2.axhline(1500, color="#DD8452", linestyle="--", linewidth=1, label="HHI 1.500 (moderada)")
ax2.set_xlabel("Provedores por município")
ax2.set_ylabel("HHI")
ax2.set_title("Número de provedores × concentração")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

###Discussão e resposta para a pergunta:

Não temos nenhum município que tenha somente um provedor de internet, então monopólio claramente declarado não há.

Frente a isso analisamos cidades com poucos provedores e alto share do líder, isso aparece somente em 7 municípios de pequeno porte, onde o share do líder médio é 86%. Podemos analisar ainda que municípios que possuam até 10 provedores tem um share do líder médio de 69,80%, que também consideramos como um alto índice de ocupação.

O gráfico da direita testa se número de provedores é bom proxy de concorrência.
Se os pontos formarem uma curva descendente bem definida, mais provedores implica
menos concentração. Se houver dispersão vertical alta para o mesmo número de
provedores, a contagem simples é enganosa — há municípios com muitos provedores
nominais e um dominante de fato.

O HHI é a medida usada por autoridades de defesa da concorrência justamente por
capturar essa diferença.

---

# Pergunta 4 — Participação da fibra e dependência de outros meios

A fibra é o padrão tecnológico atual. Municípios ainda dependentes de rádio, satélite
ou cabo metálico indicam infraestrutura defasada — o que representa tanto limitação
de serviço quanto oportunidade de substituição.

In [0]:
meio_estadual = spark.sql("""
    SELECT
        t.meio_acesso,
        SUM(f.acessos_qtd)                              AS acessos_qtd,
        COUNT(DISTINCT f.sk_municipio)                  AS municipios_qtd,
        COUNT(DISTINCT f.sk_empresa)                    AS prestadoras_qtd,
        ROUND(100.0 * SUM(f.acessos_qtd)
              / SUM(SUM(f.acessos_qtd)) OVER (), 2)     AS participacao_pct
    FROM fato_acessos f
    JOIN dim_tecnologia t ON f.sk_tecnologia = t.sk_tecnologia
    GROUP BY t.meio_acesso
    ORDER BY acessos_qtd DESC
""")

display(exibir(meio_estadual,
    numericos={"acessos_qtd": (0, ""), "municipios_qtd": (0, ""),
               "prestadoras_qtd": (0, ""), "participacao_pct": (2, " %")},
    texto=["meio_acesso"]))

In [0]:
fibra_regiao = spark.sql("""
    SELECT
        mesorregiao_nome,
        COUNT(*)                                   AS municipios_qtd,
        SUM(acessos_total_qtd)                     AS acessos_qtd,
        ROUND(100.0 * SUM(acessos_total_qtd * fibra_pct / 100)
              / SUM(acessos_total_qtd), 2)         AS fibra_agregada_pct,
        ROUND(AVG(fibra_pct), 2)                   AS fibra_media_pct,
        ROUND(MIN(fibra_pct), 2)                   AS fibra_minima_pct,
        ROUND(MAX(fibra_pct), 2)                   AS fibra_maxima_pct
    FROM mercado_municipio
    GROUP BY mesorregiao_nome
    ORDER BY fibra_agregada_pct DESC
""")

display(exibir(fibra_regiao,
    numericos={"municipios_qtd": (0, ""), "acessos_qtd": (0, ""),
               "fibra_agregada_pct": (2, " %"), "fibra_media_pct": (2, " %"),
               "fibra_minima_pct": (2, " %"), "fibra_maxima_pct": (2, " %")},
    texto=["mesorregiao_nome"]))

Municípios com menor participação de fibra, filtrando escala mínima.

In [0]:
menor_fibra = spark.sql("""
    SELECT
        m.nome_municipio, m.mesorregiao_nome, m.porte_populacional,
        m.acessos_total_qtd, m.fibra_pct, m.n_provedores_qtd,
        m.penetracao_enderecavel_pct, m.faixa_modal
    FROM mercado_municipio m
    WHERE m.domicilios_total_qtd >= 1000
    ORDER BY m.fibra_pct ASC
    LIMIT 25
""")

display(exibir(menor_fibra,
    numericos={"acessos_total_qtd": (0, ""), "fibra_pct": (2, " %"),
               "n_provedores_qtd": (0, ""),
               "penetracao_enderecavel_pct": (2, " %")},
    texto=["nome_municipio", "mesorregiao_nome", "porte_populacional",
           "faixa_modal"]))

In [0]:
pdf = fibra_regiao.toPandas()

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(pdf["mesorregiao_nome"], pdf["fibra_agregada_pct"], color="#55A868")
ax.set_xlabel("Participação da fibra (%)")
ax.set_title("Penetração de fibra por mesorregião — RS")
ax.set_xlim(0, 100)
ax.invert_yaxis()
for i, v in enumerate(pdf["fibra_agregada_pct"]):
    ax.text(v, i, f" {v:.1f}%".replace(".", ","), va="center", fontsize=9)
plt.tight_layout()
plt.show()

###Discussão e resposta para a pergunta:

Hoje temos uma alta participação de fibra óptica nos municípios ocupando 81,11% dos atendimentos.

Poucos municípios, com menos de 5 mil habitantes possuem ainda pouca ocupação de fibra óptica com níveis de 14,02% até menos de 50%.

A diferença entre participação agregada e média simples por região indica se a fibra
está concentrada nos municípios maiores. Agregada muito acima da média significa que
os grandes centros puxam o número, enquanto o município típico da região tem cobertura
menor.

Dependência de satélite tende a indicar áreas remotas sem alternativa terrestre;
dependência de rádio indica cobertura de baixo custo em áreas rurais ou periféricas;
cabo metálico indica infraestrutura legada de operadora incumbente.

---

# Pergunta 5 — Base de velocidade defasada e potencial de upgrade

`defasagem_pct` mede a participação de acessos abaixo de 34 Mbps. Municípios com
defasagem alta têm base instalada em planos antigos — mercado maduro para upgrade,
e possivelmente com infraestrutura substituível.

A leitura combinada com `fibra_pct` distingue dois casos: defasagem com fibra alta
sugere plano barato sobre rede boa (upgrade comercial); defasagem com fibra baixa
sugere limitação de infraestrutura (upgrade exige investimento em rede).

In [0]:
faixa_estadual = spark.sql("""
    SELECT
        v.faixa_velocidade,
        v.ordem_faixa,
        v.e_defasada,
        SUM(f.acessos_qtd)                          AS acessos_qtd,
        ROUND(100.0 * SUM(f.acessos_qtd)
              / SUM(SUM(f.acessos_qtd)) OVER (), 2) AS participacao_pct
    FROM fato_acessos f
    JOIN dim_faixa_velocidade v ON f.sk_faixa = v.sk_faixa
    GROUP BY v.faixa_velocidade, v.ordem_faixa, v.e_defasada
    ORDER BY v.ordem_faixa
""")

display(exibir(faixa_estadual,
    numericos={"ordem_faixa": (0, ""), "acessos_qtd": (0, ""),
               "participacao_pct": (2, " %")},
    texto=["faixa_velocidade", "e_defasada"]))

In [0]:
defasados = spark.sql("""
    SELECT
        nome_municipio, mesorregiao_nome, porte_populacional,
        domicilios_total_qtd, acessos_total_qtd,
        defasagem_pct, fibra_pct, faixa_modal,
        n_provedores_qtd, concentracao, pib_per_capita_brl,
        CASE
            WHEN fibra_pct >= 70 THEN 'Upgrade comercial'
            WHEN fibra_pct >= 40 THEN 'Misto'
            ELSE 'Requer investimento em rede'
        END AS tipo_oportunidade
    FROM mercado_municipio
    WHERE domicilios_total_qtd >= 1000
    ORDER BY defasagem_pct DESC
    LIMIT 30
""")

display(exibir(defasados,
    numericos={"domicilios_total_qtd": (0, ""), "acessos_total_qtd": (0, ""),
               "defasagem_pct": (2, " %"), "fibra_pct": (2, " %"),
               "n_provedores_qtd": (0, ""), "pib_per_capita_brl": (2, "")},
    texto=["nome_municipio", "mesorregiao_nome", "porte_populacional",
           "faixa_modal", "concentracao", "tipo_oportunidade"]))

In [0]:
pdf = spark.sql("""
    SELECT defasagem_pct, fibra_pct, acessos_total_qtd, porte_populacional
    FROM mercado_municipio
    WHERE defasagem_pct IS NOT NULL AND fibra_pct IS NOT NULL
""").toPandas()

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(pdf["fibra_pct"], pdf["defasagem_pct"],
                s=pdf["acessos_total_qtd"] / 400 + 8,
                alpha=0.45, color="#4C72B0", edgecolors="none")
ax.axvline(40, color="#C44E52", linestyle="--", linewidth=1, alpha=0.7)
ax.axhline(pdf["defasagem_pct"].median(), color="#DD8452", linestyle="--",
           linewidth=1, alpha=0.7)
ax.set_xlabel("Participação da fibra (%)")
ax.set_ylabel("Acessos abaixo de 34 Mbps (%)")
ax.set_title("Defasagem de velocidade × cobertura de fibra\n(tamanho do ponto = volume de acessos)")
plt.tight_layout()
plt.show()

###Discussão e resposta para a pergunta:

93,69% já estão classificados como sendo acima de 34Mbps, o que denota de uma alta ocupação com o topo da faixa medida.

O quadrante superior direito — defasagem alta com fibra alta — é o mais interessante
comercialmente: a rede já existe, os assinantes estão em planos antigos, e o upgrade
não exige investimento em infraestrutura.

O quadrante superior esquerdo — defasagem alta com fibra baixa — indica municípios
onde a limitação é física. Ali a oportunidade existe, mas o custo de entrada é maior.

---

# Pergunta 6 — Relação entre penetração, PIB per capita e setor econômico

Duas hipóteses a testar:

1. Municípios mais ricos têm penetração maior — poder aquisitivo viabiliza a
   contratação
2. O perfil econômico importa — municípios de serviços seriam mais conectados que os
   agropecuários, por dependerem mais de infraestrutura digital

A correlação de Pearson mede a primeira. A comparação entre setores dominantes,
a segunda.

In [0]:
base_corr = spark.sql("""
    SELECT
        penetracao_enderecavel_pct,
        pib_per_capita_brl,
        fibra_pct,
        defasagem_pct,
        n_provedores_qtd,
        densidade_demografica_hab_km2,
        populacao_residente_hab
    FROM mercado_municipio
    WHERE penetracao_enderecavel_pct IS NOT NULL
      AND pib_per_capita_brl IS NOT NULL
""")

print("Correlacao de Pearson com a penetracao enderecavel:\n")
for coluna in ["pib_per_capita_brl", "fibra_pct", "n_provedores_qtd",
               "densidade_demografica_hab_km2", "populacao_residente_hab"]:
    r = base_corr.stat.corr("penetracao_enderecavel_pct", coluna)
    print(f"  {coluna:<32} r = {r:+.4f}")

In [0]:
por_setor = spark.sql("""
    SELECT
        setor_dominante,
        COUNT(*)                                   AS municipios_qtd,
        SUM(domicilios_total_qtd)                  AS domicilios_qtd,
        SUM(acessos_total_qtd)                     AS acessos_qtd,
        ROUND(100.0 * SUM(acessos_total_qtd)
              / SUM(domicilios_total_qtd), 2)      AS penetracao_agregada_pct,
        ROUND(AVG(penetracao_enderecavel_pct), 2)  AS penetracao_media_pct,
        ROUND(PERCENTILE(penetracao_enderecavel_pct, 0.5), 2) AS penetracao_mediana_pct,
        ROUND(AVG(pib_per_capita_brl), 2)          AS pib_per_capita_medio_brl,
        ROUND(AVG(fibra_pct), 2)                   AS fibra_media_pct,
        ROUND(AVG(n_provedores_qtd), 1)            AS provedores_media_qtd
    FROM mercado_municipio
    GROUP BY setor_dominante
    ORDER BY penetracao_media_pct DESC
""")

display(exibir(por_setor,
    numericos={"municipios_qtd": (0, ""), "domicilios_qtd": (0, ""),
               "acessos_qtd": (0, ""), "penetracao_agregada_pct": (2, " %"),
               "penetracao_media_pct": (2, " %"),
               "penetracao_mediana_pct": (2, " %"),
               "pib_per_capita_medio_brl": (2, ""),
               "fibra_media_pct": (2, " %"), "provedores_media_qtd": (1, "")},
    texto=["setor_dominante"]))

In [0]:
pdf = spark.sql("""
    SELECT penetracao_enderecavel_pct, pib_per_capita_brl,
           setor_dominante, acessos_total_qtd
    FROM mercado_municipio
    WHERE penetracao_enderecavel_pct IS NOT NULL
      AND pib_per_capita_brl IS NOT NULL
      AND setor_dominante IS NOT NULL
""").toPandas()

print(f"Municipios no grafico: {len(pdf)}")

if len(pdf) == 0:
    print("Sem dados de setor dominante. Verificar o ano de referencia do PIB "
          "na camada Bronze - o desdobramento setorial so existe ate 2021.")
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

    cores = {"Agropecuaria": "#55A868", "Industria": "#4C72B0",
             "Servicos": "#DD8452", "Administracao publica": "#C44E52"}
    for setor, grupo in pdf.groupby("setor_dominante"):
        ax1.scatter(grupo["pib_per_capita_brl"], grupo["penetracao_enderecavel_pct"],
                    alpha=0.55, s=28, label=setor,
                    color=cores.get(setor, "#8172B3"), edgecolors="none")
    ax1.set_xscale("log")
    ax1.set_xlabel("PIB per capita (R$, escala log)")
    ax1.set_ylabel("Penetração (%)")
    ax1.set_title("Penetração × PIB per capita")
    ax1.legend(fontsize=8)

    setores = sorted(pdf["setor_dominante"].unique())
    dados = [pdf.loc[pdf["setor_dominante"] == s,
                     "penetracao_enderecavel_pct"].values for s in setores]

    try:                                   # Matplotlib 3.9 ou superior
        ax2.boxplot(dados, tick_labels=[s[:14] for s in setores], showfliers=True)
    except TypeError:                      # versoes anteriores
        ax2.boxplot(dados, labels=[s[:14] for s in setores], showfliers=True)

    ax2.set_ylabel("Penetração (%)")
    ax2.set_title("Penetração por setor econômico dominante")
    plt.setp(ax2.get_xticklabels(), rotation=15, ha="right")

    plt.tight_layout()
    plt.show()

###Discussão e resposta para a pergunta:

Percebemos que a distinção entre os setores econômicos é pequena, com uma variação de 2% entre os setores de Indústria, Administração Pública e Serviços. Somente o setor dominante de Agropecuária fica abaixo com 51,07% de penetração, e isso faz sentido pelo atuação de atendimento no campo ser mais restrito ao atendimento com fibra óptica.

Um coeficiente de correlação próximo de zero não significa ausência de relação —
pode indicar relação não linear, ou que outros fatores dominam. O gráfico de dispersão
em escala logarítmica ajuda a distinguir os dois casos.

Uma ressalva metodológica importante: o PIB per capita municipal é sensível a
distorções. Municípios pequenos com uma grande planta industrial ou usina apresentam
PIB per capita altíssimo sem que a renda das famílias acompanhe. Isso enfraquece a
variável como proxy de poder aquisitivo domiciliar.

Sobre o setor dominante, a diferença entre média e mediana revela se o resultado é
puxado por poucos casos extremos.

---

# Pergunta 7 — Municípios com maior oportunidade comercial

O índice combina três componentes em percentil, com peso igual:

- **Mercado não atendido** — domicílios sem acesso
- **Base defasada** — participação de acessos abaixo de 34 Mbps
- **Poder aquisitivo** — PIB per capita

Percentil evita que uma variável de escala maior domine. Peso igual é escolha neutra
declarada, sem base empírica neste MVP para ponderar de outra forma.

**O índice é ferramenta de priorização analítica, não recomendação de investimento.**

In [0]:
ranking = spark.sql("""
    SELECT
        nome_municipio, mesorregiao_nome, porte_populacional, perfil_ocupacao,
        domicilios_total_qtd, domicilios_sem_acesso_qtd,
        penetracao_enderecavel_pct, defasagem_pct, fibra_pct,
        pib_per_capita_brl, n_provedores_qtd, hhi, concentracao,
        p_nao_atendido, p_defasagem, p_renda, indice_oportunidade
    FROM indice_oportunidade
    ORDER BY indice_oportunidade DESC
    LIMIT 30
""")

display(exibir(ranking,
    numericos={"domicilios_total_qtd": (0, ""),
               "domicilios_sem_acesso_qtd": (0, ""),
               "penetracao_enderecavel_pct": (2, " %"),
               "defasagem_pct": (2, " %"), "fibra_pct": (2, " %"),
               "pib_per_capita_brl": (2, ""), "n_provedores_qtd": (0, ""),
               "hhi": (1, ""), "p_nao_atendido": (4, ""),
               "p_defasagem": (4, ""), "p_renda": (4, ""),
               "indice_oportunidade": (1, "")},
    texto=["nome_municipio", "mesorregiao_nome", "porte_populacional",
           "perfil_ocupacao", "concentracao"]))

O índice não considera concorrência. Um município pode ter índice alto e ser
disputado por dezenas de provedores. O recorte abaixo cruza oportunidade com
concentração: índice alto **e** poucos concorrentes.

In [0]:
oportunidade_baixa_concorrencia = spark.sql("""
    SELECT
        nome_municipio, mesorregiao_nome, porte_populacional,
        domicilios_total_qtd, domicilios_sem_acesso_qtd,
        penetracao_enderecavel_pct, defasagem_pct, fibra_pct,
        pib_per_capita_brl, n_provedores_qtd, hhi, concentracao,
        indice_oportunidade
    FROM indice_oportunidade
    WHERE indice_oportunidade >= 60
      AND n_provedores_qtd <= 10
    ORDER BY indice_oportunidade DESC
    LIMIT 25
""")

print(f"Municipios com indice >= 60 e ate 10 provedores: "
      f"{oportunidade_baixa_concorrencia.count()}")

display(exibir(oportunidade_baixa_concorrencia,
    numericos={"domicilios_total_qtd": (0, ""),
               "domicilios_sem_acesso_qtd": (0, ""),
               "penetracao_enderecavel_pct": (2, " %"),
               "defasagem_pct": (2, " %"), "fibra_pct": (2, " %"),
               "pib_per_capita_brl": (2, ""), "n_provedores_qtd": (0, ""),
               "hhi": (1, ""), "indice_oportunidade": (1, "")},
    texto=["nome_municipio", "mesorregiao_nome", "porte_populacional",
           "concentracao"]))

In [0]:
regiao_oportunidade = spark.sql("""
    SELECT
        mesorregiao_nome,
        COUNT(*)                              AS municipios_qtd,
        ROUND(AVG(indice_oportunidade), 1)    AS indice_medio,
        SUM(domicilios_sem_acesso_qtd)        AS nao_atendidos_qtd,
        ROUND(AVG(defasagem_pct), 2)          AS defasagem_media_pct,
        ROUND(AVG(pib_per_capita_brl), 2)     AS pib_per_capita_medio_brl
    FROM indice_oportunidade
    GROUP BY mesorregiao_nome
    ORDER BY indice_medio DESC
""")

display(exibir(regiao_oportunidade,
    numericos={"municipios_qtd": (0, ""), "indice_medio": (1, ""),
               "nao_atendidos_qtd": (0, ""), "defasagem_media_pct": (2, " %"),
               "pib_per_capita_medio_brl": (2, "")},
    texto=["mesorregiao_nome"]))

In [0]:
pdf = spark.sql("""
    SELECT nome_municipio, indice_oportunidade, n_provedores_qtd,
           domicilios_sem_acesso_qtd, mesorregiao_nome
    FROM indice_oportunidade
    WHERE indice_oportunidade IS NOT NULL
""").toPandas()

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(pdf["n_provedores_qtd"], pdf["indice_oportunidade"],
                s=pdf["domicilios_sem_acesso_qtd"] / 120 + 10,
                alpha=0.5, color="#4C72B0", edgecolors="none")
ax.axhline(60, color="#C44E52", linestyle="--", linewidth=1, alpha=0.7,
           label="Índice 60")
ax.axvline(10, color="#DD8452", linestyle="--", linewidth=1, alpha=0.7,
           label="10 provedores")
ax.set_xlabel("Provedores no município")
ax.set_ylabel("Índice de oportunidade")
ax.set_title("Oportunidade × concorrência\n(tamanho do ponto = domicílios não atendidos)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

###Discussão e resposta para a pergunta:

Considerando as análises realizadas, podemos observar que as oportunidades estão concentradas em municípios de pequeno porte de até 5 mil habitantes, pois possuem menos de 10 provedores ativos. 

O quadrante superior esquerdo do gráfico — índice alto com poucos provedores — reúne
os municípios mais atrativos para entrada: mercado com folga, base defasada, poder
aquisitivo razoável e concorrência ainda não consolidada.

O quadrante superior direito tem oportunidade nominal, mas exige disputa com
operadores já estabelecidos. Ali o custo de aquisição de cliente é maior.

Limitações do índice, que devem acompanhar qualquer leitura:

- Não considera custo de implantação de rede, que varia com distância, relevo e
  dispersão dos domicílios
- Não considera presença de infraestrutura de backbone ou fibra apagada disponível
- Usa PIB per capita como proxy de poder aquisitivo, sujeito a distorção em
  municípios com grande planta industrial
- É um retrato de um único período, sem tendência de crescimento


---

## Limitações da análise

**Recorte temporal único.** Todas as métricas são um retrato de um mês. Sem série
histórica não é possível medir crescimento, entrada e saída de prestadoras, nem
migração tecnológica.

**Denominador de mercado.** Domicílios recenseados do Censo 2022 é o melhor proxy
público disponível, mas não equivale a domicílios passíveis de conexão — não há dado
público de capacidade instalada de rede por município.

**Grupo econômico incompleto.** A Anatel classifica todos os provedores regionais
como `OUTROS`, 55% das linhas do RS. A análise competitiva usa CNPJ, o que resolve o
problema, mas impede consolidar prestadoras do mesmo dono sob marcas diferentes.

**Defasagem do PIB.** O PIB Municipal usa 2021 como referência, contra Censo de 2022
e Anatel de julho de 2026. O ano foi escolhido por ser o mais recente com
desdobramento setorial completo. A comparação assume estabilidade relativa entre
municípios — razoável para ordenação e percentil, menos para valores absolutos.

**PIB per capita como proxy de renda.** Municípios pequenos com uma grande planta
industrial ou usina apresentam PIB per capita alto sem que a renda das famílias
acompanhe. A variável mede produção no território, não poder aquisitivo domiciliar.

**Penetração acima de 100%.** Doze municípios permanecem acima de 100% mesmo com o
denominador ampliado. A investigação identificou duas causas — concentração
corporativa em municípios pequenos e saturação com competição intensa — mas nenhuma
é verificável diretamente com dado público.

**Local de declaração.** A Anatel registra o município do acesso conforme declarado
pela prestadora. Não há como validar externamente se a declaração corresponde ao
endereço do assinante.